In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from experiments import evaluation as ev

In [ ]:
df = ev.build_results("results")

In [ ]:
df.to_csv("results_5.csv", index=False)

In [ ]:
long_df = ev.wide_to_long_runs(df)
plt = ev.plot_runs_vs_gold_big(long_df, title="Gold vs Runs – Overview")

plt.savefig("Experiment_5.png", dpi=600, bbox_inches="tight")

In [ ]:
long_err = ev.errors_wide_to_long(df)
summary = ev.summarize_errors(long_err)
summary

In [ ]:
mean_conf_error_matrix = (
    summary[summary["type"] == "conf_error"]
    .pivot(index="run", columns="domain", values="mean")
    .sort_index()
)

mean_conf_error_matrix

In [ ]:
mean_error_matrix = (
    summary[summary["type"] == "error"]
    .pivot(index="run", columns="domain", values="mean")
    .sort_index()
)

mean_error_matrix

In [ ]:
total_per_run = (
    long_err
    .groupby(["type", "run"], as_index=False)
    .agg(
        N=("value", "count"),
        mean=("value", "mean"),
        sum=("value", "sum"),
        max=("value", "max"),
    )
    .sort_values(["type", "run"])
)
total_per_run

In [ ]:
df_totals = ev.total_error_per_run(df)
df_totals

In [ ]:
df_cat = ev.error_summary_per_category(df)
df_cat

In [ ]:
(df == 9).sum()

In [ ]:
import tiktoken

OUTPUT_CONTRACT_DEFAULT = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "A": {"severity": <int or null>, "confidence": <float 0..1 or null>, "findings": <list of strings>},
  "B": {"severity": <int or null>, "confidence": <float 0..1 or null>, "findings": <list of strings>},
  "C": {"severity": <int or null>, "confidence": <float 0..1 or null>, "findings": <list of strings>},
  "D": {"severity": <int or null>, "confidence": <float 0..1 or null>, "findings": <list of strings>},
  "E": {"severity": <int or null>, "confidence": <float 0..1 or null>, "findings": <list of strings>}
}
""".strip()

guidelines = Path("guidelines/a_guidelines.md").read_text(encoding="utf-8")

text = guidelines + "\n\n" + OUTPUT_CONTRACT_DEFAULT

## Token zählen: 


encoding = tiktoken.get_encoding("o200k_base")

tokens = encoding.encode(text)

print("GPT 5.2:", len(tokens))
import anthropic
import pandas as pd
import requests
from dotenv import load_dotenv
from anthropic import Anthropic
import os

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

response = client.messages.count_tokens(
    model="claude-opus-4-6",
    messages=[{"role": "user", "content": text}],
)

print(response.json())

In [ ]:
670 * 5

In [ ]:
df.sum

In [ ]:
import pandas as pd

DOMAINS = ["A", "B", "C", "D", "E"]
RUN = 5

# Gold-Spalten
gold_cols = DOMAINS

# Run-Spalten
run_cols = [
    f"{d}_{RUN}" for d in DOMAINS
] + [
    f"{d}_{RUN}_error" for d in DOMAINS
] + [
    f"{d}_{RUN}_conf" for d in DOMAINS
] + [
    f"{d}_{RUN}_conf_error" for d in DOMAINS
]

cols = gold_cols + run_cols

df_run5 = df[cols].copy()

print(df_run5.head())

In [ ]:
df_run5["A_5_error"].value_counts()

In [ ]:
df_run5["B_5_error"].value_counts()

In [ ]:
df_run5["C_5_error"].value_counts()

In [ ]:
df_run5["D_5_error"].value_counts()

In [ ]:
df_run5["E_5_error"].value_counts()

In [ ]:
df["E_5_error"].value_counts()

In [ ]:
df["E_1_error"].value_counts()

In [ ]:
df_5 = pd.read_csv("results/run_5.csv")

In [ ]:
df_5

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DOMAINS = ["A", "B", "C", "D", "E"]

def plot_run_vs_gold_big(
    long_df: pd.DataFrame,
    run_to_show: int,
    title: str | None = None,
):
    """
    Overview plot for ONE run:
    - 5 lines (A..E)
    - x-axis: call_nr
    - Gold as points
    - Predictions for the selected run as X
    """
    # Filter auf den gewünschten Run
    df_run = long_df[long_df["run"] == run_to_show].copy()
    if df_run.empty:
        available = sorted(long_df["run"].dropna().unique().tolist())
        raise ValueError(f"run_to_show={run_to_show} not found. Available runs: {available}")

    if title is None:
        title = f"Gold vs Run {run_to_show} (A–E)"

    fig, axes = plt.subplots(5, 1, sharex=True, figsize=(14, 10))

    for i, d in enumerate(DOMAINS):
        ax = axes[i]

        # Domain-Subset
        sub = df_run[df_run["domain"] == d].copy().dropna(subset=["call_nr"])

        # Gold: (einmal pro call) – beim Run-Subset reicht drop_duplicates auf call_nr
        gold_per_call = (
            sub.dropna(subset=["gold"])
               .sort_values(["call_nr"])
               .drop_duplicates(subset=["call_nr"])[["call_nr", "gold"]]
        )
        ax.scatter(gold_per_call["call_nr"], gold_per_call["gold"], s=40, label="Gold")

        # Predictions (nur dieser Run)
        pred = sub.dropna(subset=["pred"])
        ax.scatter(pred["call_nr"], pred["pred"], marker="x", s=40, label=f"Run {run_to_show}")

        ax.set_ylabel(d)
        ax.set_ylim(-0.2, 3.2)
        ax.grid(True)

        if i == 0:
            ax.legend(ncol=2, fontsize=9)

    axes[-1].set_xlabel("Call (call_nr)")

    plt.suptitle(title)
    plt.tight_layout()
    

    return fig

plt = plot_run_vs_gold_big(long_df, long_df_2, run_to_show="run5", title="Gold vs Run 5 (A–E)")
plt.savefig("Experiment_5.png", dpi=600, bbox_inches="tight")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DOMAINS = ["A", "B", "C", "D", "E"]

def plot_two_runs_vs_gold_big(
    long_df1: pd.DataFrame,
    long_df2: pd.DataFrame,
    run1,
    run2,
    label1="Run A",
    label2="Run B",
    title: str | None = None,
):

    df1 = long_df1[long_df1["run"] == run1].copy()
    df2 = long_df2[long_df2["run"] == run2].copy()

    if df1.empty:
        raise ValueError(f"{run1} not found in long_df1")

    if df2.empty:
        raise ValueError(f"{run2} not found in long_df2")

    if title is None:
        title = f"Gold vs {run1} vs {run2}"

    fig, axes = plt.subplots(5, 1, sharex=True, figsize=(14, 10))

    for i, d in enumerate(DOMAINS):
        ax = axes[i]

        sub1 = df1[df1["domain"] == d].copy().dropna(subset=["call_nr"])
        sub2 = df2[df2["domain"] == d].copy().dropna(subset=["call_nr"])

        # Gold (aus erstem DF)
        gold_per_call = (
            sub1.dropna(subset=["gold"])
                .sort_values(["call_nr"])
                .drop_duplicates(subset=["call_nr"])[["call_nr", "gold"]]
        )

        ax.scatter(
            gold_per_call["call_nr"],
            gold_per_call["gold"],
            s=40,
            color="black",
            label="Gold"
        )

        # Run 1
        pred1 = sub1.dropna(subset=["pred"])
        ax.scatter(
            pred1["call_nr"],
            pred1["pred"],
            marker="x",
            s=50,
            color="tab:blue",
            label=label1
        )

        # Run 2
        pred2 = sub2.dropna(subset=["pred"])
        ax.scatter(
            pred2["call_nr"],
            pred2["pred"],
            marker="x",
            s=50,
            color="tab:red",
            label=label2
        )

        ax.set_ylabel(d)
        ax.set_ylim(-0.2, 3.2)
        ax.grid(True)

        if i == 0:
            ax.legend(ncol=3, fontsize=9)

    axes[-1].set_xlabel("Call (call_nr)")

    plt.suptitle(title)
    plt.tight_layout()

    return fig

In [ ]:
long_df_2= pd.read_csv("long_results_4.csv")

In [ ]:
fig = plot_two_runs_vs_gold_big(
    long_df,
    long_df_2,
    run1="run5",
    run2="run6",
    label1="Experiment 5, run 5",
    label2="Experiment 4, run 6",
    title="Gold vs Run Comparison"
)

fig.savefig("model_comparison.png", dpi=600, bbox_inches="tight")

In [ ]:
df_run5["A_5_conf"]